# 05.10 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/10-choosing-and-saving/exercises.ipynb)

Exercise 1 is on paper.

## Exercise 1: one-standard-error rule (paper)

| model | CV AUC | CV std | interpretability |
|---|---|---|---|
| A: gradient boosting | 0.812 | 0.015 | needs SHAP |
| B: random forest | 0.803 | 0.018 | needs SHAP |
| C: logistic regression | 0.794 | 0.012 | high |
| D: decision tree | 0.761 | 0.025 | high |

Which models are tied with the best? Which would you choose for a weekly batch job explained to a marketing team? Which for a fraud screen where every 0.01 AUC is worth £50,000 a year?

*Your answer here.*

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

def expected_cost(y, p, t, c_voucher=5, c_lost=60):
    """Voucher for everyone with p >= t; a missed churner costs c_lost."""
    y, flag = np.asarray(y), np.asarray(p) >= t
    return c_voucher * flag.sum() + c_lost * ((y == 1) & ~flag).sum()

from pathlib import Path
def models_dir():
    """<repo>/models when run inside the course repo, ./models otherwise (e.g. Colab)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "download.py").exists():
            d = p / "models"; break
    else:
        d = Path("models")
    d.mkdir(exist_ok=True)
    return d

import time, lightgbm as lgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(5, shuffle=True, random_state=42)
candidates = {
    "logistic regression": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), "high: one weight per feature"),
    "decision tree (depth 3)": (DecisionTreeClassifier(max_depth=3, random_state=42), "high: 8 readable rules"),
    "KNN (k=51)": (make_pipeline(StandardScaler(), KNeighborsClassifier(51)), "low: 'similar to these customers'"),
    "random forest": (RandomForestClassifier(300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1), "medium: needs SHAP"),
    "LightGBM (tuned)": (lgb.LGBMClassifier(n_estimators=200, learning_rate=0.01, num_leaves=4,
                                            verbose=-1, random_state=42, n_jobs=1), "medium: needs SHAP"),
}

## Exercise 2: save the LightGBM instead

Fit the tuned LightGBM from `candidates`, choose its threshold by cost on cross-validated training predictions, and save it as a bundle exactly like section 5 (file name with date and test AUC). Then reload it and score the same new customer. Do the two models agree?

In [ ]:
# your code here

## Exercise 3: a scoring function

Write `score_customers(path, df)` that loads a bundle, checks every expected feature is present (raise a clear error if not), and returns a DataFrame with `p_churn` and `voucher` (True/False). Test it on the first 5 rows of `X_test`, and on a DataFrame missing `spend`.

In [ ]:
# your code here